# LIBRARY

In [34]:
import pandas as pd
import numpy as np
from scipy.spatial.distance import jensenshannon
import matplotlib.pyplot as plt
import seaborn as sns
import networkx as nx
from sklearn.preprocessing import MinMaxScaler
import json
import os
import pandas as pd

def distribution_similarity(real_traffic, gen_traffic, bins=10):

    # Normalization and Histogram
    real_hist, _ = np.histogram(real_traffic, bins=bins, density=True)
    gen_hist, _ = np.histogram(gen_traffic, bins=bins, density=True)

    # Prevent zeros by adding lower case numbers
    """Jensen-Shannon Divergence can have problems with histograms containing zeros (log(0) is undefined). 
    Therefore, a small number (1e-9) is added to the histograms to remove the zeros."""
    real_hist += 1e-9
    gen_hist += 1e-9

    # Similarity Calculation
    """Jensen-Shannon Divergence (JSD): Measures the similarity between two probability distributions 
    (0: exactly the same, 1: completely different)."""
    js = jensenshannon(real_hist, gen_hist)

    return js

In [35]:

REQUIRED_FIELDS = ["time", "src", "dst", "protocol", "length", "info"]
VALID_PROTOCOLS = ["ZigBee", "ZigBee HA"]
def validate_packet(line):
    """
    Attempt to validate a single packet line.
    Returns tuple (is_valid, error_message, packet_dict_or_None)
    """

    original_line = line.strip()

    # Skip empty lines
    if not original_line:
        return False, "Empty line", None

    # Try JSON load
    try:
        # Remove trailing commas if present:
        cleaned = original_line.rstrip(",")
        pkt = json.loads(cleaned)
    except Exception as e:
        return False, f"JSON parse error: {e}", None

    # Check required fields
    for field in REQUIRED_FIELDS:
        if field not in pkt:
            return False, f"Missing required field: {field}", pkt

    # Detect spelling mistakes
    wrong_fields = [f for f in pkt.keys() if f not in REQUIRED_FIELDS]
    if wrong_fields:
        return False, f"Unexpected field(s): {wrong_fields}", pkt

    # # Check duplicate keys (regex)
    # duplicate_keys = re.findall(r'"(\w+)":.*"(\w+)":', original_line)
    # if duplicate_keys:
    #     return False, "Duplicate key in packet", pkt

    # Time must be a valid float
    try:
        float(pkt["time"])
    except:
        return False, "Time not numeric", pkt

    # ---- Check: length is numeric ----
    try:
        int(pkt["length"])
    except:
        return False, "Length field is not numeric", pkt

    # ---- Check: protocol must be valid ----
    if pkt["protocol"] not in VALID_PROTOCOLS:
        return False, f"Invalid protocol type: {pkt['protocol']}", pkt
    return True, None, pkt


# RESULTS

In [36]:
real_traffic = []
with open(r'../Datasets/Experiment_1_one_way_communication_10_minute_input_sample.json', 'r') as file:
    for line in file:
        real_traffic.append(json.loads(line))

real_traffic  = pd.DataFrame(real_traffic)
real_traffic = real_traffic.drop(columns=["No.", "Info_clean"])


## EXPERIMENT 1

In [37]:
import pandas as pd
import numpy as np
from scipy.spatial.distance import jensenshannon

N = list(range(1,11)) # number of trial

num_pac = []
pack_size = []
time_mean = []
time_min = []
time_max = []
JS_metric = []

for n in N:
    file_path = f"../Generated_Traffic/JSON_files/RNN_Exp1_Trial_{n}_generated_10_minutes.json"
    
    with open(file_path, 'r') as f:
        raw_packets = json.load(f)

    valid = []
    invalid = []

    for pkt in raw_packets:
        line = json.dumps(pkt)
        is_valid, error, fixed = validate_packet(line)
        if is_valid:
            valid.append(fixed)
        else:
            invalid.append((pkt, error))

    generated_traffic = pd.DataFrame(valid)

    # with open(file_path, 'r') as f:
    #     generated_traffic = json.load(f)

    # generated_traffic = pd.DataFrame(generated_traffic)
    # print(generated_traffic)

    print(f"\n ================== Generated Traffic Trial Number {n} ==================")      

    print("Number of Packet:", len(generated_traffic))
    print("Average Packet Size:", (generated_traffic['length'].astype(int)).mean())
    print("Time Interval Average:", (generated_traffic['time'].astype(float)).diff().mean())
    
    print("Time Interval min:", (generated_traffic['time'].astype(float)).diff().min())
    print("Time Interval max:", (generated_traffic['time'].astype(float)).diff().max())
    
    print("Number of source:", generated_traffic['src'].nunique())
    print("Number of destination:", generated_traffic['dst'].nunique())
    
    num_pac.append(len(generated_traffic))
    pack_size.append((generated_traffic['length'].astype(int)).mean())
    time_mean.append((generated_traffic['time'].astype(float)).diff().mean())
    time_min.append((generated_traffic['time'].astype(float)).diff().min())
    time_max.append((generated_traffic['time'].astype(float)).diff().max())


    """ JENSEN SHANNON FOR TIMESTAMPT"""

    generated_traffic['time'] = pd.to_numeric( generated_traffic['time'], errors='coerce')
    real_traffic['Time'] = pd.to_numeric(real_traffic['Time'], errors='coerce')

    js_packet = distribution_similarity(real_traffic['Time'], generated_traffic['time'] )
    print(f"\nJensen-Shannon for time: {js_packet:.2f}")
    JS_metric.append(js_packet)
    

print("Number of Packet:", np.mean(num_pac),  
    "Average Packet Size", np.mean(pack_size),
    "Time Interval Average", np.mean(time_mean),
    "Time Interval min",  np.mean(time_min),
    "Time Interval max",  np.mean(time_max),
     "Average JS:", np.mean(JS_metric)
    )

print(JS_metric)



 ================== Generated Traffic Trial Number 1 ==================
Number of Packet: 123
Average Packet Size: 68.4390243902439
Time Interval Average: 1.2106287786885293
Time Interval min: -3081.7639440000003
Time Interval max: 2840.8242529999998
Number of source: 1
Number of destination: 3

Jensen-Shannon for time: 0.65

 ================== Generated Traffic Trial Number 2 ==================
Number of Packet: 122
Average Packet Size: 68.1311475409836
Time Interval Average: -0.40096419008264567
Time Interval min: -537.165692
Time Interval max: 569.610111
Number of source: 1
Number of destination: 2

Jensen-Shannon for time: 0.29

 ================== Generated Traffic Trial Number 3 ==================
Number of Packet: 119
Average Packet Size: 68.38655462184875
Time Interval Average: 1.7678905338983109
Time Interval min: -9360.419889
Time Interval max: 9692.00963
Number of source: 1
Number of destination: 2

Jensen-Shannon for time: 0.67

 ================== Generated Traffic Trial

# EXPERIMENT 2

In [38]:
real_traffic = []
with open(r'../Datasets/Experiment_2_input_sample.json', 'r') as file:
    for line in file:
        real_traffic.append(json.loads(line))

real_traffic  = pd.DataFrame(real_traffic)
real_traffic = real_traffic.drop(columns=["No.", "Info_clean"])


In [39]:
import pandas as pd
import numpy as np
from scipy.spatial.distance import jensenshannon

N = list(range(1,11)) # number of trial

num_pac = []
pack_size = []
time_mean = []
time_min = []
time_max = []
JS_metric = []

for n in N:
    file_path = f"../Generated_Traffic/JSON_files/RNN_Exp2_Trial_{n}_generated_10_minutes.json"
    
    with open(file_path, 'r') as f:
        raw_packets = json.load(f)

    valid = []
    invalid = []

    for pkt in raw_packets:
        line = json.dumps(pkt)
        is_valid, error, fixed = validate_packet(line)
        if is_valid:
            valid.append(fixed)
        else:
            invalid.append((pkt, error))

    generated_traffic = pd.DataFrame(valid)

        # --- CLEAN & SORT TIME ---
    generated_traffic['time'] = pd.to_numeric(generated_traffic['time'], errors='coerce')
    generated_traffic = generated_traffic.dropna(subset=['time'])  # remove NaN time rows
    generated_traffic = generated_traffic.sort_values(by='time').reset_index(drop=True)

    # with open(file_path, 'r') as f:
    #     generated_traffic = json.load(f)

    # generated_traffic = pd.DataFrame(generated_traffic)
    # print(generated_traffic)

    print(f"\n ================== Generated Traffic Trial Number {n} ==================")      

    print("Number of Packet:", len(generated_traffic))
    print("Average Packet Size:", (generated_traffic['length'].astype(int)).mean())
    print("Time Interval Average:", (generated_traffic['time'].astype(float)).diff().mean())
    
    print("Time Interval min:", (generated_traffic['time'].astype(float)).diff().min())
    print("Time Interval max:", (generated_traffic['time'].astype(float)).diff().max())
    
    print("Number of source:", generated_traffic['src'].nunique())
    print("Number of destination:", generated_traffic['dst'].nunique())
    
    num_pac.append(len(generated_traffic))
    pack_size.append((generated_traffic['length'].astype(int)).mean())
    time_mean.append((generated_traffic['time'].astype(float)).diff().mean())
    time_min.append((generated_traffic['time'].astype(float)).diff().min())
    time_max.append((generated_traffic['time'].astype(float)).diff().max())


    """ JENSEN SHANNON FOR TIMESTAMPT"""

    generated_traffic['time'] = pd.to_numeric( generated_traffic['time'], errors='coerce')
    real_traffic['Time'] = pd.to_numeric(real_traffic['Time'], errors='coerce')

    gen_inter_time = generated_traffic['time'].astype(float).diff()
    real_inter_time = real_traffic['Time'].astype(float).diff()

    js_packet = distribution_similarity(real_traffic['Time'], generated_traffic['time'] )
    # js_packet = distribution_similarity(real_inter_time[1:-1], gen_inter_time[1:-1])
    print(f"\nJensen-Shannon for time: {js_packet:.2f}")
    JS_metric.append(js_packet)
    

print("Number of Packet:", np.mean(num_pac),  
    "Average Packet Size", np.mean(pack_size),
    "Time Interval Average", np.mean(time_mean),
    "Time Interval min",  np.mean(time_min),
    "Time Interval max",  np.mean(time_max),
     "Average JS:", np.mean(JS_metric)
    )

print(JS_metric)




 ================== Generated Traffic Trial Number 1 ==================
Number of Packet: 201
Average Packet Size: 60.9452736318408
Time Interval Average: 25.821642045000004
Time Interval min: 0.0009929999999940264
Time Interval max: 1548.6116829999999
Number of source: 7
Number of destination: 6

Jensen-Shannon for time: 0.62

 ================== Generated Traffic Trial Number 2 ==================
Number of Packet: 198
Average Packet Size: 58.21212121212121
Time Interval Average: 24.191152274111676
Time Interval min: 0.0007429999999999382
Time Interval max: 3402.9595529999997
Number of source: 5
Number of destination: 7

Jensen-Shannon for time: 0.64

 ================== Generated Traffic Trial Number 3 ==================
Number of Packet: 197
Average Packet Size: 57.766497461928935
Time Interval Average: 27.930510362244895
Time Interval min: 0.009827999999970416
Time Interval max: 1802.99555
Number of source: 3
Number of destination: 8

Jensen-Shannon for time: 0.62

 ==============